In [2]:
print("Hello")

Hello


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [8]:
langsmith_api_key = os.getenv("langsmith_api_key")
groq_api_key = os.getenv("groq_API_key ")
os.environ["LANGSMITH_TRACING"] = "true"

In [6]:
# define langsmith client
from langsmith import Client
client = Client()

dataset_name = "LLM evaluation practice"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)
    


{'example_ids': ['7901f4cf-b917-44d0-a7fb-dd035ab71cc5',
  '3cb0a59e-05a6-4ea0-ad1e-47cc440d642a',
  '15f4d229-950d-4b2a-928a-2a68814bb3b5',
  '5e97f97e-2a14-4a7e-9a20-77c8984971a4',
  '533bfe3c-915a-4759-bb2c-4f7750c37e26'],
 'count': 5,
 'as_of': '2026-06-17T09:23:12.842254469Z'}

In [47]:
import groq
from langsmith import wrappers
groq_client = groq.Groq(
    api_key=groq_api_key
)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response=groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"


In [48]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [49]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "openai/gpt-oss-20b", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [50]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [51]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai/gpt-oss-20b"
)

View the evaluation results for experiment: 'openai/gpt-oss-20b-87ad144f' at:
https://smith.langchain.com/o/7b3e7725-00d4-45b4-8f88-4cd97583757b/datasets/213a20d7-22dc-4302-9b48-a1e814aa8685/compare?selectedSessions=7d099d62-b207-4915-84d2-50362d59c814




5it [00:06,  1.39s/it]
